<a href="https://colab.research.google.com/github/HKozuka/IDX-Exchange-Data-Science/blob/main/07_streamlit_app.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Streamlit App: Home Price Estimator

Users enter LivingArea, Beds, Baths and LotSize and get a predicted sale price from the final XGBoost model (05/06: max_depth=8, learning_rate=0.05, 500 trees, subsample=0.8, colsample_bytree=0.8).

The model uses 39 features. The app sets the four inputs and holds every other feature at its training median, so the output is the price of a typical home with those specs.

## 1. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import joblib
import numpy as np
import pandas as pd
import xgboost as xgb

BASE_PATH = '/content/drive/MyDrive/IDX Exchange Data Science Internship/Google Colab Notebooks/'
APP_DIR = f'{BASE_PATH}streamlit_app/'
os.makedirs(APP_DIR, exist_ok=True)

TARGET = 'log_ClosePrice'
SEED = 42

df_clean = pd.read_csv(f'{BASE_PATH}X_after_feat_engin.csv').drop(columns=['Unnamed: 0'], errors='ignore')
print('df_clean:', df_clean.shape)

df_clean: (307648, 47)


## 2. Feature Matrix

Same `to_xy` as 04 to 06, so the saved model sees exactly the training columns.

In [8]:
DROP = [TARGET, 'CloseDate', 'AssociationFee']  # AssociationFee is replaced by HOA_Monthly

def to_xy(df):
    X = df.drop(columns=[c for c in DROP if c in df.columns])
    X = X.select_dtypes(include='number').astype(float).replace([np.inf, -np.inf], np.nan)
    return X, df[TARGET]

X_all, y_all = to_xy(df_clean)
FEATURES = list(X_all.columns)
print(f'{len(FEATURES)} features')
print(FEATURES)

39 features
['ViewYN', 'WaterfrontYN', 'PoolPrivateYN', 'Stories', 'FireplaceYN', 'log_LivingArea', 'BathroomsTotalInteger', 'log_BuildingAreaTotal', 'BedroomsTotal', 'log_LotSizeSquareFeet', 'GarageSpaces', 'YearBuilt', 'Latitude', 'Longitude', 'PostalCode', 'PropertyAge', 'log_PropertyAge', 'ListSeason_sin', 'ListSeason_cos', 'ElemSchoolDist_m', 'MiddleSchoolDist_m', 'HighSchoolDist_m', 'SchoolRating_missing', 'SchoolScore', 'BedBathRatio', 'GeoCluster', 'HOA_Monthly', 'HasHOA', 'LotCoverage', 'SqftPerBedroom', 'Zip_ppsf_6m', 'Zip_sales_6m', 'Nbhd_ppsf_6m', 'Nbhd_logprice_est', 'Comp_ppsf', 'Comp_logprice_est', 'Comp_size_diff', 'Comp_dist_km', 'CoastDist_km']


## 3. Map App Inputs to Model Columns

The model never sees the raw inputs. It sees `log1p` versions of LivingArea and LotSize plus ratios built from them, so `add_derived()` rebuilds every column that depends on the four inputs. The check below confirms each formula reproduces `df_clean`. The same function is copied into `app.py`, so keep the two in sync.

`log_BuildingAreaTotal` stays at its median: it only loosely tracks living area (corr 0.30).

In [11]:
INPUTS = ['LivingArea', 'Beds', 'Baths', 'LotSize']

def add_derived(row, inputs, defaults):
    """Fill every model column that depends on the four inputs."""
    living, beds, baths, lot = (float(inputs[k]) for k in INPUTS)
    row['log_LivingArea'] = np.log1p(living)
    row['log_LotSizeSquareFeet'] = np.log1p(lot)
    row['BedroomsTotal'] = beds
    row['BathroomsTotalInteger'] = baths
    row['BedBathRatio'] = beds / max(baths, 1)
    row['SqftPerBedroom'] = living / max(beds, 1)
    row['LotCoverage'] = living / lot if lot > 0 else np.nan
    # Price estimates are log(ppsf) + log size, so they move with living area
    row['Comp_logprice_est'] = row['Comp_ppsf'] + row['log_LivingArea']
    row['Nbhd_logprice_est'] = row['Nbhd_ppsf_6m'] + row['log_LivingArea']
    # Size gap to comps: shift the typical gap by how far this home is from the median size
    row['Comp_size_diff'] = defaults['Comp_size_diff'] + row['log_LivingArea'] - defaults['log_LivingArea']
    return row

# Raw inputs recovered from the training columns (used for the check and the input ranges)
raw_inputs = pd.DataFrame({
    'LivingArea': np.expm1(X_all['log_LivingArea']),
    'Beds': X_all['BedroomsTotal'],
    'Baths': X_all['BathroomsTotalInteger'],
    'LotSize': np.expm1(X_all['log_LotSizeSquareFeet']),
})

# Check: rebuilding real rows from their raw inputs reproduces the training columns
EXACT = ['log_LivingArea', 'log_LotSizeSquareFeet', 'BedroomsTotal', 'BathroomsTotalInteger',
         'BedBathRatio', 'SqftPerBedroom', 'LotCoverage', 'Comp_logprice_est', 'Nbhd_logprice_est']
defaults = X_all.median().to_dict()
sample = X_all.dropna(subset=EXACT).sample(2000, random_state=SEED)
rebuilt = pd.DataFrame([add_derived(dict(r), raw_inputs.loc[i], defaults) for i, r in sample.iterrows()],
                       index=sample.index)
match = {c: float(np.isclose(rebuilt[c], sample[c], rtol=1e-4, atol=1e-4).mean()) for c in EXACT}
print(pd.Series(match, name='share matching'))
assert min(match.values()) > 0.99, 'add_derived() does not reproduce a training column'

log_LivingArea           1.0
log_LotSizeSquareFeet    1.0
BedroomsTotal            1.0
BathroomsTotalInteger    1.0
BedBathRatio             1.0
SqftPerBedroom           1.0
LotCoverage              1.0
Comp_logprice_est        1.0
Nbhd_logprice_est        1.0
Name: share matching, dtype: float64


## 4. Train and Save the Final Model

Refit on all of `df_clean` and save one bundle with the model, feature order, medians and input ranges.

In [12]:
final_model = xgb.XGBRegressor(n_estimators=500, max_depth=8, learning_rate=0.05,
                               subsample=0.8, colsample_bytree=0.8, tree_method='hist',
                               n_jobs=-1, random_state=SEED).fit(X_all, y_all)

bundle = {
    'model': final_model,
    'features': FEATURES,
    'defaults': defaults,
    'ranges': {k: (float(raw_inputs[k].quantile(0.01)), float(raw_inputs[k].quantile(0.99)))
               for k in INPUTS},
    'xgboost_version': xgb.__version__,
}
joblib.dump(bundle, f'{APP_DIR}xgb_price_bundle.joblib')
print('Saved', f'{APP_DIR}xgb_price_bundle.joblib')
print('Input ranges (1st to 99th pct):', bundle['ranges'])

# Quick check: same path the app takes, for a median home, then a bigger one
def predict_price(inputs):
    row = add_derived(dict(defaults), inputs, defaults)
    X = pd.DataFrame([row]).reindex(columns=FEATURES).astype(float)
    return float(np.expm1(final_model.predict(X)[0]))

med = {k: float(raw_inputs[k].median()) for k in INPUTS}
print(f"Median inputs {med}: ${predict_price(med):,.0f}")
print(f"Same home, +500 sq ft: ${predict_price({**med, 'LivingArea': med['LivingArea'] + 500}):,.0f}")

Saved /content/drive/MyDrive/IDX Exchange Data Science Internship/Google Colab Notebooks/streamlit_app/xgb_price_bundle.joblib
Input ranges (1st to 99th pct): {'LivingArea': (739.9999999999998, 5565.999999999996), 'Beds': (2.0, 6.0), 'Baths': (1.0, 6.0), 'LotSize': (1800.0, 241321.99999999985)}
Median inputs {'LivingArea': 1803.9999999999995, 'Beds': 3.0, 'Baths': 2.0, 'LotSize': 7242.000000000003}: $963,121
Same home, +500 sq ft: $1,014,757


## 5. Write the App

`app.py` and `requirements.txt` are saved to `streamlit_app/` next to the model.

In [13]:
%%writefile "/content/drive/MyDrive/IDX Exchange Data Science Internship/Google Colab Notebooks/streamlit_app/app.py"
"""Streamlit app: predict a single-family home's sale price from four inputs.

Run locally:  streamlit run app.py
Expects xgb_price_bundle.joblib (from section 4 of 07_streamlit_app.ipynb) in the same folder.
"""
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import streamlit as st

BUNDLE_PATH = Path(__file__).parent / 'xgb_price_bundle.joblib'


@st.cache_resource
def load_bundle(path):
    return joblib.load(path)


def add_derived(row, inputs, defaults):
    """Fill every model column that depends on the four inputs.

    Must match add_derived() in 07_streamlit_app.ipynb, which checks each formula against training data.
    """
    living, beds, baths, lot = (float(inputs[k]) for k in ['LivingArea', 'Beds', 'Baths', 'LotSize'])
    row['log_LivingArea'] = np.log1p(living)
    row['log_LotSizeSquareFeet'] = np.log1p(lot)
    row['BedroomsTotal'] = beds
    row['BathroomsTotalInteger'] = baths
    row['BedBathRatio'] = beds / max(baths, 1)
    row['SqftPerBedroom'] = living / max(beds, 1)
    row['LotCoverage'] = living / lot if lot > 0 else np.nan
    # Price estimates are log(ppsf) + log size, so they move with living area
    row['Comp_logprice_est'] = row['Comp_ppsf'] + row['log_LivingArea']
    row['Nbhd_logprice_est'] = row['Nbhd_ppsf_6m'] + row['log_LivingArea']
    # Size gap to comps: shift the typical gap by how far this home is from the median size
    row['Comp_size_diff'] = defaults['Comp_size_diff'] + row['log_LivingArea'] - defaults['log_LivingArea']
    return row


def predict_price(bundle, inputs):
    defaults = bundle['defaults']
    row = add_derived(dict(defaults), inputs, defaults)
    X = pd.DataFrame([row]).reindex(columns=bundle['features']).astype(float)
    log_pred = bundle['model'].predict(X)[0]
    return float(np.expm1(log_pred))     # model was trained on log1p(ClosePrice)


st.set_page_config(page_title='Home Price Estimator', page_icon='🏠')
st.title('Home Price Estimator')
st.caption('XGBoost trained on CRMLS single-family sales. Features not entered below '
           'are held at their training median.')

if not BUNDLE_PATH.exists():
    st.error(f'Model file not found: {BUNDLE_PATH.name}. Run the export cell in Colab and '
             'place the file next to app.py.')
    st.stop()

bundle = load_bundle(BUNDLE_PATH)

with st.form('inputs'):
    c1, c2 = st.columns(2)
    living = c1.number_input('Living area (sq ft)', min_value=200, max_value=20000, value=1800, step=50)
    lot = c2.number_input('Lot size (sq ft)', min_value=100, max_value=500000, value=6000, step=100)
    beds = c1.number_input('Bedrooms', min_value=1, max_value=15, value=3, step=1)
    baths = c2.number_input('Bathrooms', min_value=1, max_value=15, value=2, step=1)
    submitted = st.form_submit_button('Predict price')

if submitted:
    inputs = {'LivingArea': living, 'Beds': beds, 'Baths': baths, 'LotSize': lot}

    unusual = [k for k, v in inputs.items()
               if not bundle['ranges'][k][0] <= v <= bundle['ranges'][k][1]]
    if unusual:
        st.warning('Outside the range of 98% of training homes: ' + ', '.join(unusual) +
                   '. Treat this estimate with extra caution.')

    price = predict_price(bundle, inputs)
    st.metric('Predicted sale price', f'${price:,.0f}')
    # Test-set MdAPE was about 7%, so half of homes land within roughly this band
    st.caption(f'Typical range (±7%): ${price * 0.93:,.0f} to ${price * 1.07:,.0f}')


Overwriting /content/drive/MyDrive/IDX Exchange Data Science Internship/Google Colab Notebooks/streamlit_app/app.py


In [14]:
reqs = f'''streamlit
xgboost=={xgb.__version__}
scikit-learn
pandas
numpy
joblib
'''
with open(f'{APP_DIR}requirements.txt', 'w') as f:
    f.write(reqs)
print(reqs)

streamlit
xgboost==3.4.1
scikit-learn
pandas
numpy
joblib

